

# First Ensemble: ***Random Forest***

Now we're going to learn ensemble learning **by actually using it**.

### What is Random Forest?

Instead of building **one decision tree**:

```text
                Decision Tree
                     ↓
                  Prediction
```

Random Forest builds **many different decision trees**:

```text
       Tree 1 ──┐
       Tree 2 ──┤
       Tree 3 ──┤
       Tree 4 ──┼──→ Combined prediction
       ...      │
       Tree N ──┘
```

Each tree sees slightly different data/features, and their predictions are combined.

That's the basic idea of **ensemble learning**:

> Multiple models work together to produce a stronger prediction.

---

## 12.1 Import Random Forest

```python
from sklearn.ensemble import RandomForestRegressor
```

## 12.2 Build the pipeline

For Random Forest, **scaling isn't necessary**, so we'll use a separate preprocessing pipeline.

```python
numeric_transformer_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_transformer_tree, numeric_features),
    ("cat", categorical_transformer_tree, categorical_features)
])
```

Then:

```python
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])
```

### What is `n_estimators=300`?

It means:

```text
300 decision trees
       ↓
Random Forest
```

More trees generally make the model more stable, although training takes longer.

`n_jobs=-1` tells sklearn to use all available CPU cores.


# ***Gradient Boosting Regressor***

Random Forest:

```text
Tree 1 ─┐
Tree 2 ─┤
Tree 3 ─┼──→ average predictions
Tree 4 ─┤
Tree N ─┘
```

Gradient Boosting works differently:

```text
Tree 1 → errors
          ↓
Tree 2 learns errors
          ↓
Tree 3 learns remaining errors
          ↓
Tree 4 learns remaining errors
          ↓
Final prediction
```

So:

> **Random Forest = many trees built relatively independently and combined.**

> **Boosting = trees built sequentially, with later trees correcting previous errors.**

This distinction is important because **XGBoost and LightGBM are boosting algorithms**.

---

## 13.1 Build Gradient Boosting pipeline

```python id="qj9w2n"
from sklearn.ensemble import GradientBoostingRegressor

gbr_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])
```

### These three parameters matter

```text
n_estimators = 300
```

Number of trees.

```text
learning_rate = 0.05
```

How strongly each new tree contributes.

```text
max_depth = 3
```

Controls tree complexity.

We're using **reasonable starting values**, not claiming they're optimal. We'll tune them later.

# ***XGBoost 🚀***

Now we move to **XGBoost**, one of the most important algorithms for tabular ML.

### Gradient Boosting vs XGBoost

You already used:

```text
GradientBoostingRegressor
```

XGBoost follows the **same fundamental boosting idea**:

```text
Tree 1
  ↓
find errors
  ↓
Tree 2 learns from errors
  ↓
Tree 3 learns remaining errors
  ↓
...
  ↓
Final prediction
```

But XGBoost is a more sophisticated, optimized implementation with additional regularization and optimization techniques.

For this project, the important thing is:

> **Don't think of XGBoost as a completely different concept. Think of it as a powerful implementation of gradient-boosted trees.**

We'll use the same **tree-appropriate preprocessing** we used for Random Forest/Gradient Boosting.

### 14.1 Import

```python
from xgboost import XGBRegressor
```

### 14.2 Build XGBoost pipeline

```python
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])
```

### What these parameters mean

| Parameter              | Meaning                             |
| ---------------------- | ----------------------------------- |
| `n_estimators=1000`    | Maximum number of boosting trees    |
| `learning_rate=0.03`   | Contribution of each tree           |
| `max_depth=3`          | Controls tree complexity            |
| `subsample=0.8`        | Uses 80% of rows per boosting round |
| `colsample_bytree=0.8` | Uses 80% of features per tree       |
| `objective`            | Regression objective                |
| `n_jobs=-1`            | Use all CPU cores                   |

We're intentionally using a **small learning rate + many trees**, which is a common boosting strategy.

# ***LightGBM***

    LightGBM is a powerful gradient boosting framework designed for efficiency and speed, especially with large datasets. Here’s a concise breakdown to help you grasp it quickly:


⚡ ***Key Features***
* **Gradient Boosting:** Builds models sequentially, each correcting the errors of the previous one.

* **Leaf-wise Growth:** Unlike XGBoost’s level-wise tree growth, LightGBM grows trees leaf-wise, which often leads to better accuracy.

* **Speed & Memory Efficiency:** Uses histogram-based algorithms to reduce computation and memory usage.

* **Handles Large Data:** Scales well with millions of rows and high-dimensional features.

---

## 2. Create LightGBM pipeline

Use the same `preprocessor_tree` so the comparison is fair:

```python
from lightgbm import LGBMRegressor

lgbm_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])
```

### What these parameters mean

| Parameter              | Meaning                            |
| ---------------------- | ---------------------------------- |
| `n_estimators=1000`    | Up to 1000 boosting trees          |
| `learning_rate=0.03`   | Each tree makes a small correction |
| `num_leaves=31`        | Controls tree complexity           |
| `max_depth=-1`         | No explicit depth limit            |
| `subsample=0.8`        | Uses 80% of samples per iteration  |
| `colsample_bytree=0.8` | Uses 80% of features               |
| `n_jobs=-1`            | Uses all CPU cores                 |
